In [2]:
%pip install pandas

  Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl (10.0 MB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
    --------------------------------------- 0.3/12.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.6 MB 415.3 kB/s eta 0:00:30
   - -------------------------------------- 0.5/12.6 MB 415.3 kB/s eta 0:00:30
   -- ------------------------------------- 0.8/12.6 MB 594.0 kB/s eta 0:00:20
   --- ------------------------------------ 1.0/12.6 MB 712.1 kB/s eta 0:00:17
   ---- ----------------------------------- 1.3/12.6 MB 739.1 k


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd


In [6]:
df = pd.read_csv("Dataset.csv")

df.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


In [8]:
df.isnull().sum()

show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df.shape

(8790, 10)

In [11]:
features = [
    'type',
    'director',
    'country',
    'rating',
    'listed_in'
]

for feature in features:
    df[feature] = df[feature].fillna('')

In [12]:
df['combined_features'] = (
    df['type'] + ' ' +
    df['director'] + ' ' +
    df['country'] + ' ' +
    df['rating'] + ' ' +
    df['listed_in']
)

In [13]:
%pip install scikit-learn

  Using cached scikit_learn-1.9.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.18.0-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.24.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.9.0-cp314-cp314-win_amd64.whl (8.3 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached narwhals-2.24.0-py3-none-any.whl (461 kB)
Using cached scipy-1.18.0-cp314-cp314-win_amd64.whl (37.3 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ------------------------------- 1/5 [scipy]
   -------- ------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words='english'
)

tfidf_matrix = tfidf.fit_transform(
    df['combined_features']
)

In [15]:
tfidf_matrix.shape

(8790, 6569)

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

In [17]:
indices = pd.Series(
    df.index,
    index=df['title']
).drop_duplicates()

In [18]:
def recommend(title, num_recommendations=5):

    if title not in indices:
        return "Title not found in dataset."

    idx = indices[title]

    similarity_scores = list(
        enumerate(similarity_matrix[idx])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = similarity_scores[
        1:num_recommendations + 1
    ]

    movie_indices = [i[0] for i in similarity_scores]

    recommendations = df.iloc[movie_indices][
        ['title', 'type', 'listed_in', 'rating']
    ].copy()

    recommendations['similarity_score'] = [
        round(score, 3)
        for _, score in similarity_scores
    ]

    return recommendations

In [23]:
recommend("Ganglands")

,title,type,listed_in,rating,similarity_score
1183,Sentinelle,Movie,"Action & Adventure, Dramas, International Movies",TV-MA,0.847
2142,Earth and Blood,Movie,"Dramas, International Movies, Thrillers",TV-MA,0.774
6738,Lupin,TV Show,"Crime TV Shows, International TV Shows, TV Act...",TV-MA,0.685
8408,Crime Time,TV Show,"Crime TV Shows, International TV Shows, TV Act...",TV-MA,0.685
6698,Mortel,TV Show,"Crime TV Shows, International TV Shows, TV Dramas",TV-MA,0.617


In [20]:
df.head(10)

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in,combined_features
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries,Movie Kirsten Johnson United States PG-13 Docu...
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",TV Show Julien Leclercq France TV-MA Crime TV ...
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries",TV Show Mike Flanagan United States TV-MA TV D...
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies",Movie Bruno Garotti Brazil TV-PG Children & Fa...
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","Movie Haile Gerima United States TV-MA Dramas,..."
5,s9,TV Show,The Great British Baking Show,Andy Devonshire,United Kingdom,9/24/2021,2021,TV-14,9 Seasons,"British TV Shows, Reality TV",TV Show Andy Devonshire United Kingdom TV-14 B...
6,s10,Movie,The Starling,Theodore Melfi,United States,9/24/2021,2021,PG-13,104 min,"Comedies, Dramas",Movie Theodore Melfi United States PG-13 Comed...
7,s939,Movie,Motu Patlu in the Game of Zones,Suhas Kadav,India,5/1/2021,2019,TV-Y7,87 min,"Children & Family Movies, Comedies, Music & Mu...",Movie Suhas Kadav India TV-Y7 Children & Famil...
8,s13,Movie,Je Suis Karl,Christian Schwochow,Germany,9/23/2021,2021,TV-MA,127 min,"Dramas, International Movies",Movie Christian Schwochow Germany TV-MA Dramas...
9,s940,Movie,Motu Patlu in Wonderland,Suhas Kadav,India,5/1/2021,2013,TV-Y7,76 min,"Children & Family Movies, Music & Musicals",Movie Suhas Kadav India TV-Y7 Children & Famil...
